In [9]:
import os
import sys
import re

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")
from tvm.auto_scheduler.search_policy import CustomPrintState


ppath = os.environ.get("PYTHONPATH")
buildpath = os.environ.get("TVM_LIBRARY_PATH")
gdb_mode = os.environ.get("TVM_GDB_MODE")
use_ncu = os.environ.get("USE_NCU")

# breakpoint()
import util_manager
if gdb_mode == "1":
    gdb_manager = util_manager.GDBManager()
    gdb_manager.gdb_log_set()

print("="*80)
print("PYTHONPATH :", ppath)
print("TVM_LIBRARY_PATH :", buildpath)
print("USE_NCU :", use_ncu)

if buildpath.endswith("build"):
    print("DEBUG MODE")
elif buildpath.endswith("release"):
    print("RELEASE MODE")
else:
    AssertionError("Set Environment release/debug")

import numpy as np
from util_manager import PathManager, get_network, get_arg
import tvm
from tvm import relay, auto_scheduler
from tvm.contrib import graph_executor
import argparse
from tvm.auto_scheduler.feature import get_per_store_features_from_file

TARGET = tvm.target.Target("cuda")

def get_tasks(mod, params, path_manager, verbose=True, get_pkl=True):
    if get_pkl:
        tasks, task_weights = path_manager.tasks_pkl_use()
    
    if get_pkl is False or tasks is None:
        print("Extract tasks...")
        tasks, task_weights = auto_scheduler.extract_tasks(mod["main"], params, TARGET)
        if path_manager.tasks_pkl_check() is False:
            path_manager.tasks_pkl_save(tasks, task_weights)

    if verbose:
        for idx, task in enumerate(tasks):
            print("========== Task %d : %s  (workload key: %s) ==========" % (idx, task.desc, task.workload_key))
            print(task.compute_dag)
            # breakpoint()
    
    print(f"Total tasks length : {len(tasks)}")
    # breakpoint()
    return tasks, task_weights


PYTHONPATH : None
TVM_LIBRARY_PATH : /root/work/tvm-ansor/build-release
USE_NCU : None
RELEASE MODE


In [2]:
from types import SimpleNamespace


args = SimpleNamespace(
    network="resnet_18",
    batch_size = 1,
    dtype="float32",
    layout="NHWC",
    timenow=None,
    json=None
)


# 네트워크 불러오기
mod, params, input_shape, output_shape = get_network(args.network, args.batch_size, args.layout, dtype=args.dtype)


# 경로 설정
# path_manager = PathManager(network, input_shape, args, gdb_mode)
path_manager = PathManager(args.network, input_shape, args, gdb_mode, json="/root/work/tvm-ansor/gallery/logs_json/tmp.json")


# task 추출
tasks, task_weights = get_tasks(None, params, path_manager, verbose=False, get_pkl=True)

# task.desc 기준으로 정렬
tasks, task_weights = zip(*sorted(zip(tasks, task_weights), key=lambda x: x[0].desc))
search_policies = []
for idx, (task, weight) in enumerate(zip(tasks, task_weights)):
    print(f"T{idx} : {task.desc} ({weight})")
        
    search_policies.append(
                auto_scheduler.SketchPolicy(
                    task,
                    auto_scheduler.XGBModel(),
                )
            )


Getting network resnet_18...

Using json : /root/work/tvm-ansor/gallery/logs_json/tmp.json
Load tasks from /root/work/tvm-ansor/gallery/ansor_tasks_pkl/resnet_18-(1,224,224,3).pkl
Total tasks length : 24
T0 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add (1)
T1 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_1 (1)
T2 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_2 (1)
T3 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_3 (1)
T4 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu (1)
T5 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu_1 (1)
T6 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu_2 (1)
T7 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_multiply_add_nn_re_a4e88f85cf7823fc_ (1)
T8 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu (2)
T9 

In [3]:
for idx, task in enumerate(tasks):
    print(f"Task {idx}: {task.desc} : {task.workload_key}")

Task 0: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add : ["10b8215aaf2e14d47d40b4093e6f41a0", [1, 56, 56, 64], [6, 6, 64, 64], [1, 56, 56, 64], [1, 56, 56, 64]]
Task 1: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_1 : ["25577781e50c611c2e45e73c1cb3a6ca", [1, 28, 28, 128], [4, 4, 128, 128], [1, 28, 28, 128], [1, 28, 28, 128]]
Task 2: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_2 : ["7f3fee61bc3c2604395f5d343b840b7c", [1, 14, 14, 256], [4, 4, 256, 256], [1, 14, 14, 256], [1, 14, 14, 256]]
Task 3: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_3 : ["d78e8eb6021c4cdda0ad7775d10f751a", [1, 7, 7, 512], [4, 4, 512, 512], [1, 7, 7, 512], [1, 7, 7, 512]]
Task 4: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_add_nn_relu : ["6c4f6234946e16bcf9e48bdf289f9200", [1, 56, 56, 64], [6, 6, 64, 64], [1, 56, 56, 64], [1, 1, 1, 64], [1, 56, 56, 64]]
Task 5: vm_mod_fused_nn_contrib_conv2d_wino

In [4]:
"""
분석: .shared buffer size가 어떤 split factor에 의해 결정되는지 추적
같은 sketch에서 tile 값만 다른 여러 state를 만들어서 비교
"""
from tvm.auto_scheduler.search_policy import CustomPrintState

# dense_add task 사용 (가장 간단한 matmul 형태)
task_idx = 10
task = tasks[task_idx]
policy = search_policies[task_idx]

print(f"Task: {task.desc}")
print(f"Compute DAG:\n{task.compute_dag}")
print(f"Hardware: max_shared_memory={task.hardware_params.max_shared_memory_per_block}")
print("=" * 80)

# 여러 state 샘플링 (인자 없음)
states = policy.sample_initial_population()
print(f"Sampled {len(states)} states")

# 첫 번째 state의 구조 확인
s0 = states[0]
print("\n=== State 0 ===")
print(CustomPrintState(s0, delete_trivial_loop=False, dag_show=False))
print("\n=== Transform Steps ===")
for i, step in enumerate(s0.transform_steps):
    print(f"Step {i}: {step}")


Task: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu_2
Compute DAG:
p0 = PLACEHOLDER [1, 14, 14, 256]
data_pad(i0, i1, i2, i3) = tir.if_then_else(((((i1 >= 1) && (i1 < 15)) && (i2 >= 1)) && (i2 < 15)), p0[i0, (i1 - 1), (i2 - 1), i3], 0f)
input_tile(eps, nu, p, ci) = data_pad[floordiv(p, 49), ((floormod(floordiv(p, 7), 7)*2) + eps), ((floormod(p, 7)*2) + nu), ci]
B(i, j) = select(((floormod(i, 4) == 3) && (floormod(j, 4) == 3)), 1f, select(((floormod(i, 4) == 3) && (floormod(j, 4) == 2)),  ..(OMITTED).. ormod(i, 4) == 0) && (floormod(j, 4) == 1)), 0f, select(((floormod(i, 4) == 0) && (floormod(j, 4) == 0)), 1f, 0f))))))))))))))))
data_pack(eps, nu, p, ci) += ((input_tile[r_a, r_b, p, ci]*B[r_a, eps])*B[r_b, nu])
p1 = PLACEHOLDER [4, 4, 256, 256]
bgemm(eps, nu, p, co) += (data_pack[eps, nu, p, ci]*p1[eps, nu, co, ci])
A(i, j) = select(((floormod(i, 4) == 3) && (floormod(j, 2) == 1)), 1f, select(((floormod(i, 4) == 3) && (floormod(j, 2) == 0)),  ..(OMITTED)..

In [5]:
print(tasks[10].desc)
print(tasks[10].compute_dag)
print("="*80)
compute_ops_n = 0
for s_id, op in enumerate(tasks[10].compute_dag.ops):
    print(f"Stage{s_id+1}: {op}")
    if isinstance(op, tvm.te.tensor.ComputeOp):
        compute_ops_n += 1
print("="*80)
print(tasks[10].compute_dag.init_state)
print("="*80)
s = search_policies[10].sample_initial_population()[0]
for step in s.transform_steps:
    print(step)
print("="*80)
print(CustomPrintState(s, delete_trivial_loop=False, dag_show=False))

vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu_2
p0 = PLACEHOLDER [1, 14, 14, 256]
data_pad(i0, i1, i2, i3) = tir.if_then_else(((((i1 >= 1) && (i1 < 15)) && (i2 >= 1)) && (i2 < 15)), p0[i0, (i1 - 1), (i2 - 1), i3], 0f)
input_tile(eps, nu, p, ci) = data_pad[floordiv(p, 49), ((floormod(floordiv(p, 7), 7)*2) + eps), ((floormod(p, 7)*2) + nu), ci]
B(i, j) = select(((floormod(i, 4) == 3) && (floormod(j, 4) == 3)), 1f, select(((floormod(i, 4) == 3) && (floormod(j, 4) == 2)),  ..(OMITTED).. ormod(i, 4) == 0) && (floormod(j, 4) == 1)), 0f, select(((floormod(i, 4) == 0) && (floormod(j, 4) == 0)), 1f, 0f))))))))))))))))
data_pack(eps, nu, p, ci) += ((input_tile[r_a, r_b, p, ci]*B[r_a, eps])*B[r_b, nu])
p1 = PLACEHOLDER [4, 4, 256, 256]
bgemm(eps, nu, p, co) += (data_pack[eps, nu, p, ci]*p1[eps, nu, co, ci])
A(i, j) = select(((floormod(i, 4) == 3) && (floormod(j, 2) == 1)), 1f, select(((floormod(i, 4) == 3) && (floormod(j, 2) == 0)),  ..(OMITTED).. ct(((floormod(i, 4

In [6]:
print(tasks[10].desc)
print(tasks[10].compute_dag.init_state)
print("="*80)
for i, step in enumerate(s.transform_steps):
    print(f"Step {i} : {step}")
    partial_state = tvm._ffi.get_global_func("auto_scheduler.ReplayStepsPartial")(tasks[10].compute_dag, s, i)
    print(partial_state)
    print("="*80)

vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu_2
Placeholder: p0, p1, p2
for i0 (0,1)
  for i1 (0,16)
    for i2 (0,16)
      for i3 (0,256)
        data_pad = ...
for eps (0,4)
  for nu (0,4)
    for p (0,49)
      for ci (0,256)
        input_tile = ...
for i (0,4)
  for j (0,4)
    B = ...
for eps (0,4)
  for nu (0,4)
    for p (0,49)
      for ci (0,256)
        for r_a (0,4)
          for r_b (0,4)
            data_pack = ...
for eps (0,4)
  for nu (0,4)
    for p (0,49)
      for co (0,256)
        for ci (0,256)
          bgemm = ...
for i (0,4)
  for j (0,2)
    A = ...
for vh (0,2)
  for vw (0,2)
    for p (0,49)
      for co (0,256)
        for r_a (0,4)
          for r_b (0,4)
            inverse = ...
for n (0,1)
  for h (0,14)
    for w (0,14)
      for co (0,256)
        conv2d_winograd = ...
for ax0 (0,1)
  for ax1 (0,14)
    for ax2 (0,14)
      for ax3 (0,256)
        T_add = ...
for ax0 (0,1)
  for ax1 (0,14)
    for ax2 (0,14)
      for 

In [7]:
# VisitAttrs 추가 후 Step 속성 접근 테스트 (고유 타입별)
print("=== transform step 속성 접근 테스트 ===\n")
seen_types = set()
for i, step in enumerate(s.transform_steps):
    # 실제 C++ 타입 확인
    type_key = step.type_key if hasattr(step, 'type_key') else '?'
    # if type_key in seen_types:
    #     continue
    seen_types.add(type_key)
    attrs = [x for x in dir(step) if not x.startswith('_') and x not in ('handle', 'legacy_repr', 'same_as', 'type_key')]
    print(f"Step {i}: type_key={type_key}, python_type={type(step).__name__}, attrs={attrs}")
    for attr in attrs:
        try:
            val = getattr(step, attr)
            print(f"  {attr} = {val}")
        except Exception as e:
            print(f"  {attr} = ERROR: {e}")
    print()

=== transform step 속성 접근 테스트 ===

Step 0: type_key=auto_scheduler.ComputeInlineStep, python_type=Object, attrs=['stage_id']
  stage_id = 11

Step 1: type_key=auto_scheduler.ComputeInlineStep, python_type=Object, attrs=['stage_id']
  stage_id = 9

Step 2: type_key=auto_scheduler.AnnotationStep, python_type=Object, attrs=['annotation', 'iter_id', 'stage_id']
  annotation = 1
  iter_id = 0
  stage_id = 8

Step 3: type_key=auto_scheduler.AnnotationStep, python_type=Object, attrs=['annotation', 'iter_id', 'stage_id']
  annotation = 1
  iter_id = 1
  stage_id = 8

Step 4: type_key=auto_scheduler.SplitStep, python_type=Object, attrs=['extent', 'inner_to_outer', 'iter_id', 'lengths', 'stage_id']
  extent = 49
  inner_to_outer = 1
  iter_id = 2
  lengths = [1]
  stage_id = 8

Step 5: type_key=auto_scheduler.SplitStep, python_type=Object, attrs=['extent', 'inner_to_outer', 'iter_id', 'lengths', 'stage_id']
  extent = 256
  inner_to_outer = 1
  iter_id = 4
  lengths = [32]
  stage_id = 8

Step 6:

In [8]:
import re

replay_partial = tvm._ffi.get_global_func("auto_scheduler.ReplayStepsPartial")

# 각 step index 기준으로 고유 symbolic factor 이름 생성
split_meta_by_step = {}
unroll_meta_by_step = {}
for step_idx, step in enumerate(s.transform_steps):
    type_key = getattr(step, "type_key", "")
    if type_key == "auto_scheduler.SplitStep":
        lengths = list(getattr(step, "lengths", []))
        if lengths:
            split_meta_by_step[step_idx] = {
                "sym": f"split_factor_{step_idx}",
                "factor": int(lengths[-1]),
                "extent": int(getattr(step, "extent", 0)),
            }
    elif type_key == "auto_scheduler.PragmaStep":
        pragma_type = str(getattr(step, "pragma_type", ""))
        m = re.match(r"auto_unroll_max_step\$(\d+)$", pragma_type)
        if m:
            unroll_meta_by_step[step_idx] = {
                "sym": f"unroll_factor_{step_idx}",
                "factor": int(m.group(1)),
            }

split_line_re = re.compile(
    r"^(?P<lhs>\s*[^=]+?=\s*s\[[^\]]+\]\.split\((?P<iter>[A-Za-z_][A-Za-z0-9_]*)\s*,\s*factor=)(?P<factor>\d+)(?P<rhs>\)\s*)$"
)
unroll_line_re = re.compile(
    r'^(?P<prefix>\s*s\[[^\]]+\]\.pragma\(.*"auto_unroll_max_step"\s*,\s*)(?P<factor>\d+)(?P<suffix>\)\s*)$'
)


def _iter_base_name(iter_name: str) -> str:
    # inverse_p -> p, inverse_r_a -> r_a
    if "_" in iter_name:
        return iter_name.split("_", 1)[1]
    return iter_name


def symbolicize_partial_output(python_code: str, state_text: str, n_steps: int):
    active_split_steps = [idx for idx in range(n_steps) if idx in split_meta_by_step]
    active_unroll_steps = [idx for idx in range(n_steps) if idx in unroll_meta_by_step]

    split_ptr = 0
    unroll_ptr = 0
    split_records = []

    out_lines = []
    for line in python_code.split("\n"):
        m_split = split_line_re.match(line)
        if m_split and split_ptr < len(active_split_steps):
            step_idx = active_split_steps[split_ptr]
            split_ptr += 1
            meta = split_meta_by_step[step_idx]
            sym = meta["sym"]
            out_lines.append(f"{m_split.group('lhs')}{sym}{m_split.group('rhs')}")
            split_records.append(
                {
                    "iter_base": _iter_base_name(m_split.group("iter")),
                    "extent": meta["extent"],
                    "sym": sym,
                    "factor": meta["factor"],
                }
            )
            continue

        m_unroll = unroll_line_re.match(line)
        if m_unroll and unroll_ptr < len(active_unroll_steps):
            step_idx = active_unroll_steps[unroll_ptr]
            unroll_ptr += 1
            sym = unroll_meta_by_step[step_idx]["sym"]
            out_lines.append(f"{m_unroll.group('prefix')}{sym}{m_unroll.group('suffix')}")
            continue

        out_lines.append(line)

    symbolic_state = state_text
    for rec in split_records:
        base = re.escape(rec["iter_base"])
        sym = rec["sym"]
        extent = rec["extent"]

        # state 표기: for p.0 (0,49//split_factor_x), for p.1 (0,split_factor_x)
        symbolic_state = re.sub(
            rf"(for\s+{base}\.0\s*\(0,)([^\)]+)(\))",
            lambda m, extent=extent, sym=sym: f"{m.group(1)}{extent}//{sym}{m.group(3)}",
            symbolic_state,
        )
        symbolic_state = re.sub(
            rf"(for\s+{base}\.1\s*\(0,)([^\)]+)(\))",
            lambda m, sym=sym: f"{m.group(1)}{sym}{m.group(3)}",
            symbolic_state,
        )

    return "\n".join(out_lines), symbolic_state


print(replay_partial(tasks[10].compute_dag, s, 0))
for i in range(1, len(s.transform_steps) + 1):
    partial_state = replay_partial(tasks[10].compute_dag, s, i)
    python_code = str(tasks[10].compute_dag.print_python_code_from_state(partial_state))
    symbolic_python_code, symbolic_state = symbolicize_partial_output(python_code, str(partial_state), i)

    print(s.transform_steps[:i])
    print("\n".join(symbolic_python_code.split("\n")[compute_ops_n:]))
    print(symbolic_state)




Placeholder: p0, p1, p2
for i0 (0,1)
  for i1 (0,16)
    for i2 (0,16)
      for i3 (0,256)
        data_pad = ...
for eps (0,4)
  for nu (0,4)
    for p (0,49)
      for ci (0,256)
        input_tile = ...
for i (0,4)
  for j (0,4)
    B = ...
for eps (0,4)
  for nu (0,4)
    for p (0,49)
      for ci (0,256)
        for r_a (0,4)
          for r_b (0,4)
            data_pack = ...
for eps (0,4)
  for nu (0,4)
    for p (0,49)
      for co (0,256)
        for ci (0,256)
          bgemm = ...
for i (0,4)
  for j (0,2)
    A = ...
for vh (0,2)
  for vw (0,2)
    for p (0,49)
      for co (0,256)
        for r_a (0,4)
          for r_b (0,4)
            inverse = ...
for n (0,1)
  for h (0,14)
    for w (0,14)
      for co (0,256)
        conv2d_winograd = ...
for ax0 (0,1)
  for ax1 (0,14)
    for ax2 (0,14)
      for ax3 (0,256)
        T_add = ...
for ax0 (0,1)
  for ax1 (0,14)
    for ax2 (0,14)
      for ax3 (0,256)
        T_relu = ...

[auto_scheduler.ComputeInlineStep(0x10e18bf8)

In [24]:
"""
GetPerStoreFeaturesWorkerFunc의 TIR pass 파이프라인을 순차 적용하며 TIR 출력 확인
C++ feature.cc의 GPU pass 순서를 그대로 재현:
  Phase 0: InjectPrefetch → StorageFlatten(64)
  Phase 1: NarrowDataType(32) → Simplify → VectorizeLoop → InjectVirtualThread → StorageRewrite → Simplify
  검증:    VerifyGPUCode
"""
import tvm
from tvm import tir

task = tasks[10]
print(f"Task: {task.desc}")
print(f"Hardware: shared_mem={task.hardware_params.max_shared_memory_per_block}, "
      f"threads={task.hardware_params.max_threads_per_block}, "
      f"vector_bytes={task.hardware_params.vector_unit_bytes}, "
      f"max_vthread={task.hardware_params.max_vthread_extent}")
print("=" * 80)

# 1) ApplySteps: State → te::Schedule
sch, tensors = task.compute_dag.apply_steps_from_state(s)

# 2) normalize (C++에서는 normalize_for_feature_extraction 사용)
sch = sch.normalize()

# 3) ScheduleToModule: te::Schedule → TIR IRModule
mod = tvm.driver.lower(sch, args=tensors, name="main")

print(">>> [0] ScheduleToModule 직후 (원본 TIR)")
print(mod.script())
print("=" * 80)

# GPU TIR pass 파이프라인 (feature.cc와 동일한 순서)
pass_sequence = [
    ("InjectPrefetch",       tir.transform.InjectPrefetch()),
    ("StorageFlatten(64)",   tir.transform.StorageFlatten(64)),
    ("NarrowDataType(32)",   tir.transform.NarrowDataType(32)),
    ("Simplify #1",          tir.transform.Simplify()),
    ("VectorizeLoop",        tir.transform.VectorizeLoop(True)),
    ("InjectVirtualThread",  tir.transform.InjectVirtualThread()),
    ("StorageRewrite",       tir.transform.StorageRewrite()),
    ("Simplify #2",          tir.transform.Simplify()),
]

for i, (name, p) in enumerate(pass_sequence):
    try:
        mod = p(mod)
        print(f">>> [{i+1}] {name} 적용 후")
        print(mod.script())
        print("=" * 80)
    except Exception as e:
        print(f">>> [{i+1}] {name} 적용 중 에러: {e}")
        print("=" * 80)
        break

# VerifyGPUCode (분석 pass - 에러만 출력)
print(">>> [9] VerifyGPUCode 검증")
gpu_params = {
    "max_shared_memory_per_block": task.hardware_params.max_shared_memory_per_block,
    "max_local_memory_per_block":  task.hardware_params.max_local_memory_per_block,
    "max_threads_per_block":       task.hardware_params.max_threads_per_block,
    "max_vector_bytes":            task.hardware_params.vector_unit_bytes,
    "max_vthread":                 task.hardware_params.max_vthread_extent,
}
try:
    prim_func = mod["main"]
    result = tvm.tir.analysis.verify_gpu_code(prim_func, gpu_params)
    print(f"  결과: {result}")
    if result:
        print("  ⚠️ GPU 제약 위반 발견!")
    else:
        print("  ✅ GPU 제약 통과")
except Exception as e:
    print(f"  에러: {e}")
print("=" * 80)

# 최종 Simplify (feature extraction 전 마지막 단계)
print(">>> [10] 최종 Simplify")
try:
    final_mod = tir.transform.Simplify()(mod)
    print(final_mod.script())
except Exception as e:
    print(f"  에러: {e}")
print("=" * 80)

Task: vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu_2
Hardware: shared_mem=49152, threads=1024, vector_bytes=16, max_vthread=8
>>> [0] ScheduleToModule 직후 (원본 TIR)
# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def main(p0: T.Buffer((1, 14, 14, 256), "float32"), p1: T.Buffer((4, 4, 256, 256), "float32"), p2: T.Buffer((1, 1, 1, 256), "float32"), T_relu: T.Buffer((1, 14, 14, 256), "float32")):
        T.func_attr({"from_legacy_te_schedule": T.bool(True), "tir.noalias": T.bool(True)})
        data_pack = T.allocate([200704], "float32", "global")
        bgemm = T.allocate([200704], "float32", "global")
        data_pack_1 = T.Buffer((200704,), data=data_pack)
        with T.launch_thread("blockIdx.x", 392) as blockIdx_x:
            input_tile = T.allocate([16], "float32", "local")
            threadIdx_x = T.launch_thread("threadIdx.x", 32)
            input_tile_1 = T.Buffer((16,), data=i